# 09 — Factor-Augmented Switching Regression: Does the Rest of the Data Help Price?

**Hypothesis under test (framework v3.1 §6).** The data we collect beyond price — trade flow, the L2 book, the perpetual basis, the option surface, activity and liquidity — helps identify the regimes and forecast price.

**Status: exploratory** (about 15 sessions; the binding test is gated at 60–90 days). This is step B of the v3.1 nested sequence: static factors (A) feeding a **price** equation, before any switching factor dynamics.

Measurements are compressed into one factor per feature family (training-only clock adjustment, standardization and PCA; `modeling/factors.py`). Two channels:

$$
z_t = \mu_k + \beta_k D^c_{t-1} + \boldsymbol\gamma_k^\top \mathbf F^{dir}_{t-1} + \sum_{j=1}^{3}\rho_j z_{t-j} + \sigma_k\varepsilon_t, \qquad
\mathbf F^{state}_t \mid R_t = k \sim \mathcal N(\mathbf m_k, \operatorname{diag}\mathbf s_k^2)^{\lambda}, \qquad R_t = k \ \text{(HSMM)}
$$

- $\mathbf F^{dir}$: **directional** families (flow, book imbalance, basis, skew) — signed pressure that can predict the sign of the next index return; state-specific slopes $\boldsymbol\gamma_k$.
- $\mathbf F^{state}$: **state** families (activity, volatility, liquidity, option level) — symmetric measurements that cannot predict the sign but may identify the regime; a masked emission block tempered by $\lambda \in \{0.25, 1\}$ so it cannot take over the states (Notebook 08 Part D).
- $D^c$: session-VWAP displacement (framework v2.2); $z_t = r_t/\sigma^c_{t-1}$, index returns.

**The decision metric is the out-of-sample log score of the next index return**, never the fit of the features. The hypothesis is supported only if adding factors improves that score over the Notebook 08 model, and switching ($K=2$) improves on the same factors without switching ($K=1$).

| Part | Question |
|---|---|
| A | Factors: coverage, explained share, loadings, stability |
| B | Without regimes: does any family predict the next return beyond lags and $D$? |
| C | With regimes ($K=2$): which families improve the price score; does the state block help or take over? |
| D | How many regimes does the factor-augmented model support? |
| E | What do the states look like ($\beta_k$, $\boldsymbol\gamma_k$, $\sigma_k$, durations)? |
| F | Is the slope separation more than regime-free paths produce (random-sign null, real features)? |

Runtime on a desktop with numba: about 30–60 minutes. `NB09_K_MAX`, `NB09_NULL_PATHS`, `NB09_NULL_ASSETS` change the workload.

In [ ]:
from pathlib import Path
import os
import sys
import time
import warnings

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

load_dotenv()
DB_PATH = Path(os.environ["DERIBIT_DATA_ROOT"]).expanduser().resolve() / "database" / "research.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)
con.execute("SET TimeZone = 'UTC'")

PROJECT_ROOT = Path(os.getenv("DERIBIT_PROJECT_ROOT", Path.cwd())).resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "data_engineering").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "data_engineering").exists():
    raise RuntimeError("Repository not found: start Jupyter inside the repository or set DERIBIT_PROJECT_ROOT.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
for _m in [m for m in sys.modules if m.split(".")[0] == "data_engineering"]:
    del sys.modules[_m]

from data_engineering.features import regime_inputs as ri
from data_engineering.modeling.clock_baseline import ClockBaselineSet, FeatureSpec
from data_engineering.modeling import factors as fx
from data_engineering.modeling.hsmm.engine import HSMMSpec, fit, episodes, predictive_scores, ENGINE_VERSION, USE_NUMBA
from data_engineering.modeling.hsmm.design import build_design
from data_engineering.modeling.hsmm.selection import select_K
from data_engineering.modeling.hsmm.baselines import hac_t
from data_engineering.modeling.hsmm.null import simulate_returns, p_value
if tuple(int(x) for x in ri.REGIME_INPUT_VERSION.split(".")) < (0, 3) or ENGINE_VERSION < "0.1.2":
    raise RuntimeError("Old src modules: git pull and restart the kernel.")

CONTRACT_ASSETS = ["AVAX", "BTC", "ETH", "HYPE", "SOL", "XRP"]
K_MAX = int(os.getenv("NB09_K_MAX", "4"))
NULL_PATHS = int(os.getenv("NB09_NULL_PATHS", "19"))
NULL_ASSETS = os.getenv("NB09_NULL_ASSETS", "BTC,XRP,ETH,SOL").split(",")
FIT_KW = {"n_init": 2, "max_iter": 200, "tol": 1e-5}
LAGS = ["z1", "z2", "z3"]
print(f"src {PROJECT_ROOT / 'src'} | engine {ENGINE_VERSION} | factors {fx.FACTOR_VERSION} | numba {'on' if USE_NUMBA else 'OFF'} | "
      f"K = 1..{K_MAX} | null {NULL_PATHS} paths on {NULL_ASSETS}")

## 1. Load, sessions and clock factor for returns

In [ ]:
cols = set(con.execute("DESCRIBE time_series_feature_state").df()["column_name"])
base_cols = ["state_id", "state_time", "underlying", "session_id", "return_5m", "futures_index_price", "is_weekend",
             "in_settlement_window", "fit_ok", "option_atm_quote_ok",
             "vwap", "volume_usd", "trade_count", "futures_tradable_mid_price", "perp_integrity_ok",
             "known_exchange_event", "integrity_flag_trades_outside_bbo"]
factor_cols = sorted({fd.column for fam in fx.FAMILIES for fd in fam.features} |
                     {"trade_flow_imbalance", "book_twa_obi_20", "book_twa_bid_depth_20", "book_twa_ask_depth_20",
                      "option_iv_atm_forward", "realized_vol_1h_annualized"})
want = [c for c in dict.fromkeys(base_cols + factor_cols) if c in cols]
print("Not in time_series_feature_state (skipped):", sorted(set(base_cols + factor_cols) - cols - {
    "tfi_30m", "obi_20_30m", "book_depth_20_usd", "atm_iv_quote", "iv_minus_rv"}))
df = con.execute(f"SELECT {', '.join(want)} FROM time_series_feature_state ORDER BY underlying, state_time").df()
df["state_time"] = pd.to_datetime(df["state_time"], utc=True)
for c in ["is_weekend", "in_settlement_window", "fit_ok", "option_atm_quote_ok", "perp_integrity_ok",
          "known_exchange_event", "integrity_flag_trades_outside_bbo"]:
    if c in df:
        s = df[c]
        if s.dtype == object:
            s = s.map({"True": True, "False": False, True: True, False: False})
        df[c] = s.fillna(False).astype(bool)
df = df[df["underlying"].isin(CONTRACT_ASSETS)].reset_index(drop=True)
df["r"] = pd.to_numeric(df["return_5m"], errors="coerce")
df["session_date"] = ri.session_key(df["state_time"])
dates = sorted(df.loc[df["r"].notna(), "session_date"].unique())
TRAIN_DATES = set(dates[: len(dates) // 2])
train_mask = df["session_date"].isin(TRAIN_DATES)
clock = ClockBaselineSet.fit(df[["state_id", "state_time", "underlying", "is_weekend", "return_5m"]], train_mask,
                             specs=(FeatureSpec("return_5m", "return_5m", transform="log_abs", scale="none"),))
df["f_clock"] = clock.transform(df[["state_id", "state_time", "underlying", "is_weekend", "return_5m"]])[
    "return_5m__clock_vol_factor"].to_numpy()
df["f_clock"] = df["f_clock"].fillna(1.0)
print(f"Rows {len(df):,} | assets {sorted(df.underlying.unique())} | sessions {len(dates)} | "
      f"train {min(TRAIN_DATES)} → {max(TRAIN_DATES)} | test {min(set(dates) - TRAIN_DATES)} → {max(dates)}")

# Part A — Factors

Each feature is clock-adjusted (Fourier baseline with weekend terms, fitted on the training sessions), standardized with training moments and winsorized at ±5; each family is reduced to its first principal component on the training rows, signed so that the family's anchor feature loads positively. Missing values count as 0 (the training mean, "no information"); a factor is missing only when every feature of its family is. `loading_stability` is $|\cos|$ between the loadings fitted on the first and second half of the training rows (1 = identical). Under the v3.1 rule a family whose loadings are unstable (below about 0.8) is not ready for the switching model.

In [ ]:
t0 = time.perf_counter()
FS = fx.build_factors(df, train_mask)
print(f"Factors built in {time.perf_counter() - t0:.0f}s | missing raw columns: {FS.missing_columns} | "
      f"clock fits skipped: {len(FS.clock_skipped)}")
display(FS.summary.pivot(index=["channel", "family"], columns="underlying", values="explained_pc1").round(2))
print("Coverage (share of states with the factor):")
display(FS.summary.pivot(index=["channel", "family"], columns="underlying", values="coverage").round(2))
print("Loading stability (first vs second half of training):")
display(FS.summary.pivot(index=["channel", "family"], columns="underlying", values="loading_stability").round(2))
print("Loadings:")
display(FS.loadings.pivot_table(index=["family", "feature"], columns="underlying", values="loading").round(2))
corrs = [FS.factors[train_mask & (df["underlying"] == a)].corr() for a in sorted(df["underlying"].unique())]
print("Median (over assets) correlation between family factors, training rows:")
display(pd.concat(corrs).groupby(level=0).median().reindex(FS.factors.columns)[FS.factors.columns].round(2))
DIR = [c for c in FS.directional().columns if FS.factors[c].notna().any()]
STATE = [c for c in FS.state().columns if FS.factors[c].notna().any()]
print("Directional factors:", DIR, "| state factors:", STATE)

## 2. Designs

One design per asset with the session-VWAP anchor, the directional factors as lagged regressors (known at $t-1$ for row $t$) and the state factors as the masked block. All models below share the same rows, so their scores are compared on identical test bars.

In [ ]:
designs = {}
for asset, g in df.groupby("underlying"):
    idx = g.index
    g = g.reset_index(drop=True)
    E = FS.factors.loc[idx, DIR].reset_index(drop=True)
    B = FS.factors.loc[idx, STATE].reset_index(drop=True)
    d = build_design(g, g["f_clock"].to_numpy(), extra=E, block=B.fillna(0.0), block_mask=B.notna())
    designs[asset] = (d, g)
display(pd.DataFrame({a: {"rows": len(d.frame), "segments": len(d.starts), "sd_D": d.frame.D.std(),
                          **{f"sd_{c}": float(np.std(d.columns[:, d.col_names.index(c)])) for c in DIR}}
                      for a, (d, g) in designs.items()}).T.round(3))

# Part B and C — Out-of-sample price score by family

Every model is fitted on the training half and scored on the test half by the one-step predictive log density of $z_t$ (filtering uses the block where present; only $z_t$ is scored). Gains are in 1/1000 nats per bar, with Newey–West $t$ on the per-bar score difference.

- **Base:** lags + $D$, no regimes ($K=1$).
- **NB08:** lags + $D$, $K=2$ (the Notebook 08 model, re-fitted here).
- **+family, $K=1$ / $K=2$:** one directional family added.
- **+all dir:** every directional family.
- **block λ:** NB08 model plus the state block with weight $\lambda$.

The hypothesis needs (i) a factor model beating **NB08** and **Base** with $|t| > 2$ on several assets, and (ii) its $K=2$ version beating its own $K=1$ version (otherwise the factors help, but not through regimes).

In [ ]:
def model_list():
    m = [("Base K1", ["const", "D"], 1, False, 1.0), ("NB08 K2", ["const", "D"], 2, False, 1.0)]
    for c in DIR:
        m += [(f"+{c[2:]} K1", ["const", "D", c], 1, False, 1.0), (f"+{c[2:]} K2", ["const", "D", c], 2, False, 1.0)]
    m += [("+all dir K1", ["const", "D"] + DIR, 1, False, 1.0), ("+all dir K2", ["const", "D"] + DIR, 2, False, 1.0)]
    if STATE:
        m += [("NB08 + block λ=0.25", ["const", "D"], 2, True, 0.25), ("NB08 + block λ=1", ["const", "D"], 2, True, 1.0),
              ("+all dir + block λ=0.25 K2", ["const", "D"] + DIR, 2, True, 0.25)]
    return m


score_rows, fitted = [], {}
for asset, (d, g) in designs.items():
    t0 = time.perf_counter()
    tr = d.frame["session_date"].isin(TRAIN_DATES).to_numpy()
    scores = {}
    for name, xs, K, blk, lam in model_list():
        res = fit(d.data(xs=xs, xc=LAGS, rows=tr, with_block=blk, block_weight=lam), HSMMSpec(K=K, seed=1, **FIT_KW))
        scores[name] = predictive_scores(res.params, d.data(xs=xs, xc=LAGS, rows=~tr, with_block=blk, block_weight=lam))
        fitted[(asset, name)] = res
    for name, s in scores.items():
        for ref in ["Base K1", "NB08 K2"]:
            dlt = s - scores[ref]
            score_rows.append({"underlying": asset, "model": name, "vs": ref, "gain_x1000": 1000 * dlt.mean(), "t": hac_t(dlt)})
        k1 = name.replace("K2", "K1")
        if name.endswith("K2") and k1 in scores and k1 != name:
            dlt = s - scores[k1]
            score_rows.append({"underlying": asset, "model": name, "vs": "same columns K1", "gain_x1000": 1000 * dlt.mean(),
                               "t": hac_t(dlt)})
    print(f"{asset}: {len(scores)} models, {time.perf_counter() - t0:.0f}s")
scores_tab = pd.DataFrame(score_rows)
order = [m[0] for m in model_list()]
for ref in ["Base K1", "NB08 K2", "same columns K1"]:
    sub = scores_tab[scores_tab.vs == ref]
    print(f"\nGain vs {ref} (x1000 nats per bar):")
    display(sub.pivot(index="model", columns="underlying", values="gain_x1000").reindex([o for o in order if o in set(sub.model)]).round(1))
    print(f"Newey-West t vs {ref}:")
    display(sub.pivot(index="model", columns="underlying", values="t").reindex([o for o in order if o in set(sub.model)]).round(2))

# Part D — How many regimes with the factors?

The same selection rule as Notebook 08 (blocked expanding-window CV over sessions, one-standard-error rule, at least 20 episodes per state), applied to the **+all dir** model, next to the Notebook 08 model on the same rows.

In [ ]:
sel_rows, K_SEL = [], {}
for asset, (d, g) in designs.items():
    t0 = time.perf_counter()
    for label, xs in [("NB08", ["const", "D"]), ("+all dir", ["const", "D"] + DIR)]:
        K_sel, tab, info = select_K(d, K_values=tuple(range(1, K_MAX + 1)), spec_kw=FIT_KW, verbose=False,
                                    data_kw={"xs": xs, "xc": LAGS})
        sel_rows.append(tab.reset_index().assign(underlying=asset, model=label))
        K_SEL[(asset, label)] = K_sel
    print(f"{asset}: selected K NB08 = {K_SEL[(asset, 'NB08')]}, +all dir = {K_SEL[(asset, '+all dir')]} "
          f"({time.perf_counter() - t0:.0f}s)")
k_sel = pd.concat(sel_rows, ignore_index=True)
k_sel["oos_vs_NB08_K1"] = k_sel.oos_logscore - k_sel.apply(
    lambda r: k_sel[(k_sel.underlying == r.underlying) & (k_sel.model == "NB08") & (k_sel.K == 1)].oos_logscore.iloc[0], axis=1)
print("Total out-of-sample log score relative to the NB08 model at K = 1 (nats, CV test sessions):")
display(k_sel.pivot_table(index=["model", "K"], columns="underlying", values="oos_vs_NB08_K1").round(1))
print("Episodes per state (support needs >= 20):")
display(k_sel[k_sel.model == "+all dir"].pivot(index="K", columns="underlying", values="episodes_per_state"))

# Part E — States of the factor-augmented model (full sample, $K=2$)

Description only. States ordered by $\beta_D$. A factor matters for the regime if its slope differs between states; it matters for price in general if its slope is similar and non-zero in both (then $K=1$ with the factor is enough).

In [ ]:
state_rows, ep_rows = [], []
for asset, (d, g) in designs.items():
    xs = ["const", "D"] + DIR
    data = d.data(xs=xs, xc=LAGS)
    res = fit(data, HSMMSpec(K=2, n_init=4, max_iter=300, seed=1))
    p = res.params
    for k in range(2):
        row = {"underlying": asset, "state": k, "occupancy": float(res.gamma[:, k].mean()), "sigma": p.sigma[k],
               "mean_duration": float(p.mean_duration()[k]), "nu": p.nu}
        row.update({f"b_{n}": p.beta_s[k, j] for j, n in enumerate(xs)})
        state_rows.append(row)
    ep = pd.DataFrame(episodes(res.gamma, data))
    ep_rows.append(ep.groupby("state").agg(episodes=("length", "size"), median_bars=("length", "median")).assign(underlying=asset).reset_index())
states = pd.DataFrame(state_rows)
display(states.round(3))
display(pd.concat(ep_rows).set_index(["underlying", "state"]))

# Part F — Regime-free null for the factor slopes

Random signs on the real return magnitudes (`sign` null): the real features, factors, volatility path and missing-data pattern are kept, and any relation between the factors or $D$ and the **sign** of the next return is destroyed. The session VWAP is rebuilt on each path (framework v2.2 §6). Statistic: the separation of the state slopes over $D$ and the directional factors at $K=2$,

$$
\Delta = \Bigl\| \boldsymbol\beta_{1} - \boldsymbol\beta_{0} \Bigr\|_2, \qquad \boldsymbol\beta_k = (\beta_{k,D}, \boldsymbol\gamma_k)
$$

One-sided empirical $p = (1 + \#\{\Delta^* \ge \hat\Delta\})/(1 + N)$; with 19 paths the smallest attainable $p$ is 0.05. The AR-bootstrap null is not used here: it would have to simulate the features jointly with returns.

In [ ]:
def slope_sep(design, xs):
    res = fit(design.data(xs=xs, xc=LAGS), HSMMSpec(K=2, seed=3, **FIT_KW))
    b = res.params.beta_s[:, 1:]
    return float(np.linalg.norm(b[1] - b[0])), b


null_rows = []
for asset in NULL_ASSETS:
    if asset not in designs:
        continue
    d, g = designs[asset]
    xs = ["const", "D"] + DIR
    idx = df.index[df.underlying == asset]
    E = FS.factors.loc[idx, DIR].reset_index(drop=True)
    t0 = time.perf_counter()
    real, b_real = slope_sep(d, xs)
    rng = np.random.default_rng(sum(map(ord, asset)))
    draws = []
    for _ in range(NULL_PATHS):
        r_star = simulate_returns(g, g["f_clock"].to_numpy(), "sign", rng)
        dn = build_design(g, g["f_clock"].to_numpy(), returns=r_star, extra=E, rng=rng)
        draws.append(slope_sep(dn, xs)[0])
    draws = np.array(draws)
    null_rows.append({"underlying": asset, "separation_real": real, "null_p50": np.median(draws),
                      "null_p95": np.quantile(draws, 0.95), "p_value": p_value(real, draws),
                      "seconds": time.perf_counter() - t0})
    print(f"{asset}: real {real:.3f} | null median {np.median(draws):.3f} | p {p_value(real, draws):.3f}")
null_test = pd.DataFrame(null_rows)
display(null_test.round(3))

## Interpretation checklist

| Result | Implication |
|---|---|
| A: a family with loading stability < 0.8 or coverage < 0.5 | Not ready for the switching model; drop it or wait for more data. |
| B: a family $K=1$ beats Base with $\lvert t\rvert > 2$ | That data predicts price without regimes — useful, but it is not regime evidence. |
| C: a factor $K=2$ beats NB08 **and** its own $K=1$ with $\lvert t\rvert > 2$ on several assets | **Supports the hypothesis**: the extra data identifies regimes that matter for price. |
| C: block λ = 1 worse than λ = 0.25 or than NB08 | The state block takes over the states (as in Notebook 08); keep it tempered or move it to the hazard. |
| C: nothing beats NB08 | At this sample size the extra data does not help price; the v3 factor layer is not justified yet. |
| D: selected $K$ higher with factors than without | Factors reveal states that price alone cannot. |
| E: $\gamma$ similar across states | The factor matters, but not as a regime discriminator. |
| F: $p \le 0.05$ | The slope separation exceeds what regime-free sign paths produce with the same features. |

The test half is about 7 sessions: treat $\lvert t\rvert$ between 2 and 3 as provisional.

In [ ]:
nb09_report = {"factor_summary": FS.summary, "loadings": FS.loadings, "scores": scores_tab, "k_selection": k_sel,
               "states": states, "null_test": null_test}
for k, v in nb09_report.items():
    print(f"  {k:<15} {len(v):>6,} rows")
con.close()
print("DuckDB connection closed.")